# T31-机构行为监测 - 核心算法模块

## 1. 模块概述

核心算法模块包含机构行为监测所需的各类分析算法，包括：

- **RFY Premium (评级基准收益率溢价)**: 衡量交易收益率相对评级的偏离
- **Herding Index (羊群效应指数)**: 基于LSV模型计算市场羊群效应
- **Duration of Net Flow (久期净流向)**: 计算机构久期配置倾向
- **概率模拟算法**: 模拟机构间交易对手方
- **风险定价指标**: 期限利差和信用利差计算

## 2. RFY Premium (评级基准收益率溢价)

RFY Premium 用于衡量交易收益率相对于同一评级基准收益率的偏离程度，反映机构的风险偏好。

In [ ]:
import pandas as pd
import numpy as np
from typing import Optional, List, Dict, Any

class RFYPremiumCalculator:
    """RFY Premium 计算器"""
    
    def __init__(self):
        self.benchmark_yield = None
    
    def calculate_benchmark_yield(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算评级基准收益率
        
        Args:
            df: 包含 DT, yield, deal_amount, rating 列的数据
        
        Returns:
            DataFrame: 每日每评级的基准收益率
        """
        # 清洗数据
        df = df.dropna(subset=['yield', 'deal_amount', 'rating'])
        df = df[df['rating'] != '']
        
        # 计算加权收益率
        df['weighted_yield_sum'] = df['yield'] * df['deal_amount']
        
        # 按日期和评级分组
        benchmark = df.groupby(['DT', 'rating']).agg(
            benchmark_yield=('weighted_yield_sum', 'sum'),
            total_deal_amount=('deal_amount', 'sum')
        ).reset_index()
        
        # 计算加权平均
        benchmark['benchmark_yield'] = benchmark['benchmark_yield'] / benchmark['total_deal_amount']
        
        self.benchmark_yield = benchmark
        return benchmark
    
    def calculate_rfi_premium(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算每笔交易的RFY Premium
        
        Args:
            df: 交易数据（需先计算benchmark_yield）
        
        Returns:
            DataFrame: 包含RFY Premium的数据
        """
        if self.benchmark_yield is None:
            self.calculate_benchmark_yield(df)
        
        # 合并基准收益率
        df = df.merge(
            self.benchmark_yield[['DT', 'rating', 'benchmark_yield']],
            on=['DT', 'rating'],
            how='left'
        )
        
        # 计算RFY Premium
        df['rfi_premium'] = df['yield'] - df['benchmark_yield']
        
        return df
    
    def calculate_market_rfi_index(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算每日市场平均RFI指数
        
        Args:
            df: 包含RFY Premium的数据
        
        Returns:
            DataFrame: 每日市场RFI指数
        """
        # 计算加权RFI Premium
        df['weighted_rfi_premium'] = df['rfi_premium'] * df['deal_amount']
        
        # 按日期聚合
        market_rfi = df.groupby('DT').agg(
            total_weighted_premium=('weighted_rfi_premium', 'sum'),
            total_deal_amount=('deal_amount', 'sum')
        ).reset_index()
        
        # 计算市场平均
        market_rfi['market_rfi_index'] = (
            market_rfi['total_weighted_premium'] / market_rfi['total_deal_amount']
        )
        
        return market_rfi[['DT', 'market_rfi_index']]

# 创建RFY计算器
rfy_calculator = RFYPremiumCalculator()
print("RFY Premium计算器已创建")

## 3. Herding Index (羊群效应指数)

基于LSV (Lakonishok, Shleifer, Vishny) 模型计算市场羊群效应。

公式: H = |p_i - p| - AF

其中:
- p_i: 某债券的买入比例
- p: 市场平均买入比例
- AF: 调整因子（避免随机买卖导致的虚假羊群效应）

In [ ]:
class HerdingIndexCalculator:
    """羊群效应指数计算器"""
    
    def __init__(self):
        pass
    
    def calculate_daily_herding_index(self, daily_trades: pd.DataFrame) -> float:
        """
        计算单日羊群效应指数
        
        Args:
            daily_trades: 单日交易数据，包含 TRADE_CODE, PARTY_ID_B, PARTY_ID_S, deal_amount
        
        Returns:
            float: 羊群效应指数
        """
        # 计算每个机构在每只债券上的净买入额
        buys = daily_trades.groupby(['TRADE_CODE', 'PARTY_ID_B'])['deal_amount'].sum().reset_index()
        buys.rename(columns={'PARTY_ID_B': 'party', 'deal_amount': 'buy_amount'}, inplace=True)
        
        sells = daily_trades.groupby(['TRADE_CODE', 'PARTY_ID_S'])['deal_amount'].sum().reset_index()
        sells.rename(columns={'PARTY_ID_S': 'party', 'deal_amount': 'sell_amount'}, inplace=True)
        
        # 合并买卖数据
        net_positions = pd.merge(buys, sells, on=['TRADE_CODE', 'party'], how='outer').fillna(0)
        net_positions['net_amount'] = net_positions['buy_amount'] - net_positions['sell_amount']
        
        # 统计每只债券的净买方和净卖方数量
        bond_stats = net_positions.groupby('TRADE_CODE').agg(
            buyers=('net_amount', lambda x: (x > 0).sum()),
            sellers=('net_amount', lambda x: (x < 0).sum())
        ).reset_index()
        
        bond_stats['N_i'] = bond_stats['buyers'] + bond_stats['sellers']
        bond_stats = bond_stats[bond_stats['N_i'] > 0]
        
        if bond_stats.empty:
            return 0.0
        
        # 计算 p_i 和 p
        bond_stats['p_i'] = bond_stats['buyers'] / bond_stats['N_i']
        p_t = bond_stats['buyers'].sum() / bond_stats['N_i'].sum()
        
        # 计算调整因子 AF（简化版本）
        # 真实的AF计算涉及二项分布期望，这里用标准误差近似
        bond_stats['AF_i'] = np.sqrt(p_t * (1 - p_t) / bond_stats['N_i'])
        
        # 计算 Herding Index
        bond_stats['H_i'] = np.abs(bond_stats['p_i'] - p_t) - bond_stats['AF_i']
        
        # 返回市场平均羊群效应
        return bond_stats['H_i'].mean()
    
    def calculate_herding_time_series(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算羊群效应指数时序
        
        Args:
            df: 交易数据，包含 DT, TRADE_CODE, PARTY_ID_B, PARTY_ID_S, deal_amount
        
        Returns:
            DataFrame: 每日羊群效应指数
        """
        results = []
        
        for dt in df['DT'].unique():
            daily_trades = df[df['DT'] == dt]
            herding_index = self.calculate_daily_herding_index(daily_trades)
            results.append({'DT': dt, 'herding_index': herding_index})
        
        return pd.DataFrame(results)

# 创建羊群效应计算器
herding_calculator = HerdingIndexCalculator()
print("羊群效应指数计算器已创建")

## 4. Duration of Net Flow (久期净流向)

计算机构买入/卖出交易的加权久期，反映机构的久期配置倾向。

In [ ]:
import re

class DurationFlowCalculator:
    """久期净流向计算器"""
    
    @staticmethod
    def parse_tenor_to_years(tenor_str: str) -> float:
        """
        将期限字符串转换为年数
        
        Args:
            tenor_str: 期限字符串
        
        Returns:
            float: 年数
        """
        if pd.isna(tenor_str):
            return np.nan
        
        tenor_str = str(tenor_str)
        
        if 'Y' in tenor_str:
            match = re.match(r'(\d+\.?\d*)Y', tenor_str)
            if match:
                return float(match.group(1))
        
        if 'D' in tenor_str:
            match = re.match(r'(\d+\.?\d*)D', tenor_str)
            if match:
                return float(match.group(1)) / 365
        
        return np.nan
    
    def calculate_daily_duration_flow(self, daily_trades: pd.DataFrame) -> float:
        """
        计算单日久期净流向
        
        Args:
            daily_trades: 单日交易数据，包含 DIRECTION, TRANSACTION_AMOUNT, duration_approx
        
        Returns:
            float: 净流向久期
        """
        # 分离买卖
        buys = daily_trades[daily_trades['DIRECTION'] == '买入']
        sells = daily_trades[daily_trades['DIRECTION'] == '卖出']
        
        # 计算买入加权久期
        if not buys.empty and buys['TRANSACTION_AMOUNT'].sum() > 0:
            weighted_duration_buy = (
                (buys['duration_approx'] * buys['TRANSACTION_AMOUNT']).sum() /
                buys['TRANSACTION_AMOUNT'].sum()
            )
        else:
            weighted_duration_buy = 0
        
        # 计算卖出加权久期
        if not sells.empty and sells['TRANSACTION_AMOUNT'].sum() > 0:
            weighted_duration_sell = (
                (sells['duration_approx'] * sells['TRANSACTION_AMOUNT']).sum() /
                sells['TRANSACTION_AMOUNT'].sum()
            )
        else:
            weighted_duration_sell = 0
        
        return weighted_duration_buy - weighted_duration_sell
    
    def calculate_duration_flow_time_series(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算久期净流向时序
        
        Args:
            df: 交易数据
        
        Returns:
            DataFrame: 每日久期净流向
        """
        # 转换期限
        df = df.copy()
        df['duration_approx'] = df['tenor'].apply(self.parse_tenor_to_years)
        df = df.dropna(subset=['duration_approx'])
        
        results = []
        for dt in df['DT'].unique():
            daily_trades = df[df['DT'] == dt]
            duration_flow = self.calculate_daily_duration_flow(daily_trades)
            results.append({'DT': dt, 'duration_flow': duration_flow})
        
        return pd.DataFrame(results)

# 创建久期流向计算器
duration_calculator = DurationFlowCalculator()
print("久期净流向计算器已创建")

## 5. 机构交易概率模拟

基于历史统计数据模拟机构间交易对手方，用于生成机构间交易矩阵。

In [ ]:
class InstitutionTradeSimulator:
    """机构交易概率模拟器"""
    
    def __init__(self):
        self.stats_df = None
    
    def load_stats(self, stats_df: pd.DataFrame):
        """
        加载机构交易统计数据
        
        Args:
            stats_df: 包含机构类型、期限、债券类型、买入卖出量的统计
        """
        self.stats_df = stats_df
    
    def get_probability_distribution(self, tenor: str, bond_type: str, 
                                     trade_side: str) -> pd.DataFrame:
        """
        计算买/卖方概率分布
        
        Args:
            tenor: 期限
            bond_type: 债券类型
            trade_side: 'buy' 或 'sell'
        
        Returns:
            DataFrame: 机构类型和概率
        """
        if self.stats_df is None:
            raise ValueError("请先加载统计数据")
        
        # 筛选匹配的切片
        market_slice = self.stats_df[
            (self.stats_df['期限'] == tenor) &
            (self.stats_df['债券类型'] == bond_type)
        ]
        
        col = 'buy_vol' if trade_side == 'buy' else 'sell_vol'
        total_vol = market_slice[col].sum()
        
        if total_vol == 0:
            # 回退到全市场分布
            market_slice = self.stats_df
            total_vol = market_slice[col].sum()
            if total_vol == 0:
                return None
        
        result = market_slice[['机构类型', col]].copy()
        result['prob'] = result[col] / total_vol
        
        return result[['机构类型', 'prob']]
    
    def assign_counterparty(self, tenor: str, bond_type: str) -> Dict[str, str]:
        """
        为交易分配买卖双方机构类型
        
        Args:
            tenor: 期限
            bond_type: 债券类型
        
        Returns:
            Dict: 包含买方和卖方机构类型
        """
        # 获取买方概率分布
        buy_dist = self.get_probability_distribution(tenor, bond_type, 'buy')
        
        # 获取卖方概率分布
        sell_dist = self.get_probability_distribution(tenor, bond_type, 'sell')
        
        # 抽样
        buyer = np.random.choice(buy_dist['机构类型'], p=buy_dist['prob']) if buy_dist is not None else '未知'
        seller = np.random.choice(sell_dist['机构类型'], p=sell_dist['prob']) if sell_dist is not None else '未知'
        
        return {'party_type_b': buyer, 'party_type_s': seller}
    
    def simulate_trade_matrix(self, dealtinfo_df: pd.DataFrame, 
                              stats_df: pd.DataFrame) -> pd.DataFrame:
        """
        模拟生成机构间交易矩阵
        
        Args:
            dealtinfo_df: 交易明细数据
            stats_df: 机构统计数据
        
        Returns:
            DataFrame: 机构间交易矩阵
        """
        self.load_stats(stats_df)
        
        # 为每笔交易分配机构
        results = []
        for _, row in dealtinfo_df.iterrows():
            parties = self.assign_counterparty(
                row.get('tenor_category', '1-3Y'),
                row.get('bond_type', '中期票据')
            )
            results.append({
                'party_type_b': parties['party_type_b'],
                'party_type_s': parties['party_type_s'],
                'transaction_amount': row['transaction_amount']
            })
        
        df_simulated = pd.DataFrame(results)
        
        # 生成交易矩阵
        trade_matrix = df_simulated.pivot_table(
            index='party_type_s',
            columns='party_type_b',
            values='transaction_amount',
            aggfunc='sum'
        ).fillna(0)
        
        return trade_matrix

# 创建交易模拟器
trade_simulator = InstitutionTradeSimulator()
print("机构交易概率模拟器已创建")

## 6. 市场核心行为指数计算

整合RFY、羊群效应和久期流向，计算市场核心行为指数。

In [ ]:
class MarketBehaviorIndexCalculator:
    """市场核心行为指数计算器"""
    
    def __init__(self):
        self.rfy_calculator = RFYPremiumCalculator()
        self.herding_calculator = HerdingIndexCalculator()
        self.duration_calculator = DurationFlowCalculator()
    
    def calculate_all_indices(self, credit_trades: pd.DataFrame,
                              all_trades: pd.DataFrame) -> pd.DataFrame:
        """
        计算所有市场行为指数
        
        Args:
            credit_trades: 信用债交易数据（用于RFY）
            all_trades: 全部交易数据（用于久期流向）
        
        Returns:
            DataFrame: 整合的市场行为指数
        """
        # 计算RFI指数
        credit_trades = self.rfy_calculator.calculate_rfi_premium(credit_trades)
        rfi_daily = self.rfy_calculator.calculate_market_rfi_index(credit_trades)
        
        # 计算久期流向
        duration_daily = self.duration_calculator.calculate_duration_flow_time_series(all_trades)
        
        # 合并结果
        df_final = pd.merge(rfi_daily, duration_daily, on='DT', how='outer')
        df_final = df_final.fillna(0)
        
        # 重命名列
        df_final.rename(columns={
            'DT': 'date',
            'market_rfi_index': 'rfi_premium',
            'duration_flow': 'duration_net_flow'
        }, inplace=True)
        
        return df_final
    
    def calculate_institution_scorecard(self, institution_type: str,
                                        credit_trades: pd.DataFrame,
                                        all_trades: pd.DataFrame) -> Dict[str, Any]:
        """
        计算机构行为记分卡
        
        Args:
            institution_type: 机构类型
            credit_trades: 信用债交易数据
            all_trades: 全部交易数据
        
        Returns:
            Dict: 机构行为评分
        """
        # 这里简化为模拟值，实际需要筛选机构数据
        return {
            'institution_type': institution_type,
            'rfi_score': np.random.uniform(-0.1, 0.1),
            'rfi_change': np.random.uniform(-0.02, 0.02),
            'herding_score': np.random.uniform(0, 0.5),
            'herding_change': np.random.uniform(-0.1, 0.1),
            'duration_flow_score': np.random.uniform(2, 5),
            'duration_flow_change': np.random.uniform(-0.5, 0.5)
        }

# 创建市场行为指数计算器
behavior_calculator = MarketBehaviorIndexCalculator()
print("市场核心行为指数计算器已创建")

## 7. 风险定价指标计算

In [ ]:
class RiskPricingCalculator:
    """风险定价指标计算器"""
    
    @staticmethod
    def get_approx_tenor(tenor_str: str) -> Optional[int]:
        """
        获取近似期限（年）
        
        Args:
            tenor_str: 期限字符串
        
        Returns:
            int: 期限年数
        """
        if tenor_str is None:
            return None
        if '10Y' in tenor_str:
            return 10
        if '1Y' in tenor_str:
            return 1
        if '3Y' in tenor_str:
            return 3
        if '5Y' in tenor_str:
            return 5
        return None
    
    def calculate_term_spread(self, df_rate_bonds: pd.DataFrame) -> pd.DataFrame:
        """
        计算期限利差（10Y - 1Y）
        
        Args:
            df_rate_bonds: 国债交易数据
        
        Returns:
            DataFrame: 期限利差时序
        """
        df = df_rate_bonds.copy()
        df['tenor_approx'] = df['tenor'].apply(self.get_approx_tenor)
        df = df.dropna(subset=['tenor_approx', 'yield'])
        
        # 计算每日各期限平均收益率
        daily_yields = df.groupby(['DT', 'tenor_approx'])['yield'].mean().unstack()
        
        # 计算期限利差
        if 10 in daily_yields.columns and 1 in daily_yields.columns:
            daily_yields['term_spread'] = daily_yields[10] - daily_yields[1]
        else:
            daily_yields['term_spread'] = np.nan
        
        return daily_yields[['term_spread']].reset_index()
    
    def calculate_credit_spread(self, df_credit: pd.DataFrame, 
                                df_rate_3y: pd.DataFrame) -> pd.DataFrame:
        """
        计算信用利差（AA+ 3Y中票 - 3Y国债）
        
        Args:
            df_credit: 信用债交易数据
            df_rate_3y: 3年期国债收益率
        
        Returns:
            DataFrame: 信用利差时序
        """
        # 计算每日中票收益率
        daily_mtn = df_credit.groupby('DT')['yield'].mean().reset_index()
        daily_mtn.rename(columns={'yield': 'mtn_yield'}, inplace=True)
        
        # 计算每日国债收益率
        daily_gov = df_rate_3y.groupby('DT')['yield'].mean().reset_index()
        daily_gov.rename(columns={'yield': 'gov_yield'}, inplace=True)
        
        # 合并计算利差
        df_merged = pd.merge(daily_mtn, daily_gov, on='DT', how='inner')
        df_merged['credit_spread'] = (df_merged['mtn_yield'] - df_merged['gov_yield']) * 100  # BP
        
        return df_merged[['DT', 'credit_spread']]
    
    def calculate_all_spreads(self, df_rate: pd.DataFrame, 
                              df_credit: pd.DataFrame) -> pd.DataFrame:
        """
        计算所有利差指标
        
        Args:
            df_rate: 国债数据
            df_credit: 信用债数据
        
        Returns:
            DataFrame: 整合的利差数据
        """
        # 计算期限利差
        df_term_spread = self.calculate_term_spread(df_rate)
        
        # 筛选3年期国债
        df_rate_3y = df_rate[df_rate['tenor'].str.contains('3Y', na=False)]
        
        # 计算信用利差
        df_credit_spread = self.calculate_credit_spread(df_credit, df_rate_3y)
        
        # 合并
        df_final = pd.merge(df_term_spread, df_credit_spread, on='DT', how='outer')
        df_final.rename(columns={'DT': 'date'}, inplace=True)
        df_final = df_final.sort_values('date').fillna(method='ffill')
        
        return df_final

# 创建风险定价计算器
risk_calculator = RiskPricingCalculator()
print("风险定价指标计算器已创建")

## 8. 热点分析算法

In [ ]:
class HotspotAnalyzer:
    """机构交易热点分析器"""
    
    def __init__(self):
        pass
    
    def find_institution_hotspots(self, df: pd.DataFrame, 
                                  institution_type: str,
                                  top_n: int = 10) -> pd.DataFrame:
        """
        找出机构交易热点债券
        
        Args:
            df: 交易数据，包含 SEC_NAME, transaction_amount
            institution_type: 机构类型
            top_n: 返回前N只
        
        Returns:
            DataFrame: 热点债券列表
        """
        # 按债券汇总
        bond_stats = df.groupby('SEC_NAME').agg({
            'transaction_amount': 'sum',
            'TRADE_CODE': 'count'
        }).reset_index()
        
        bond_stats.rename(columns={
            'SEC_NAME': 'name',
            'transaction_amount': 'total_amount',
            'TRADE_CODE': 'trade_count'
        }, inplace=True)
        
        # 模拟买卖分配
        np.random.seed(42)
        bond_stats['buy_amount'] = bond_stats['total_amount'] * np.random.uniform(0.4, 0.6, len(bond_stats))
        bond_stats['sell_amount'] = bond_stats['total_amount'] - bond_stats['buy_amount']
        bond_stats['net_buy'] = bond_stats['buy_amount'] - bond_stats['sell_amount']
        
        # 排序取前N
        df_hot = bond_stats.sort_values('total_amount', ascending=False).head(top_n)
        df_hot['type'] = f'{institution_type}热点'
        df_hot['value'] = df_hot['total_amount']
        
        return df_hot
    
    def calculate_concentration_ratio(self, df: pd.DataFrame, 
                                      top_n: int = 10) -> float:
        """
        计算交易集中度
        
        Args:
            df: 交易数据
            top_n: 前N只债券
        
        Returns:
            float: 集中度比率
        """
        total_amount = df['transaction_amount'].sum()
        
        bond_totals = df.groupby('SEC_NAME')['transaction_amount'].sum()
        top_amounts = bond_totals.nlargest(top_n).sum()
        
        return top_amounts / total_amount if total_amount > 0 else 0

# 创建热点分析器
hotspot_analyzer = HotspotAnalyzer()
print("机构交易热点分析器已创建")

## 9. 算法模块总结

In [ ]:
# 模块总结
print("=" * 60)
print("核心算法模块组件列表")
print("=" * 60)
print("\n1. RFYPremiumCalculator - RFY Premium计算器")
print("   - calculate_benchmark_yield(): 计算评级基准收益率")
print("   - calculate_rfi_premium(): 计算RFY Premium")
print("   - calculate_market_rfi_index(): 计算市场RFI指数")
print("\n2. HerdingIndexCalculator - 羊群效应指数计算器")
print("   - calculate_daily_herding_index(): 计算单日羊群效应")
print("   - calculate_herding_time_series(): 计算时序数据")
print("\n3. DurationFlowCalculator - 久期净流向计算器")
print("   - parse_tenor_to_years(): 期限转换为年数")
print("   - calculate_daily_duration_flow(): 计算单日久期流向")
print("   - calculate_duration_flow_time_series(): 计算时序数据")
print("\n4. InstitutionTradeSimulator - 机构交易概率模拟器")
print("   - get_probability_distribution(): 获取概率分布")
print("   - assign_counterparty(): 分配交易对手")
print("   - simulate_trade_matrix(): 模拟交易矩阵")
print("\n5. MarketBehaviorIndexCalculator - 市场核心行为指数计算器")
print("   - calculate_all_indices(): 计算所有指数")
print("   - calculate_institution_scorecard(): 计算机构记分卡")
print("\n6. RiskPricingCalculator - 风险定价指标计算器")
print("   - calculate_term_spread(): 计算期限利差")
print("   - calculate_credit_spread(): 计算信用利差")
print("   - calculate_all_spreads(): 计算所有利差")
print("\n7. HotspotAnalyzer - 机构交易热点分析器")
print("   - find_institution_hotspots(): 找出热点债券")
print("   - calculate_concentration_ratio(): 计算集中度")
print("\n" + "=" * 60)
print("核心算法模块加载完成")
print("=" * 60)